# Archived student modelling cells

Author: Paul Trassaert. Original modelling code is retained for provenance, with no outputs, private connector, interview template or private API documentation.

**Evaluation caveat:** the reported 0.95 macro-F1 and 0.9981 AUC came from an internal split reused for early stopping. The separately supplied labelled test file was not evaluated. This historical workflow is superseded by `../train.py`.

Requires original feature exports and optional XGBoost/SHAP/plotting dependencies. Feature column names differ from the portable refactor; no equivalence of results is implied.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.model_selection import StratifiedKFold, train_test_split, RandomizedSearchCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_curve, auc
final_train = pd.read_csv("../data/train_features.csv")
final_test = pd.read_csv("../data/test_features.csv")
final_valid = pd.read_csv("../data/unlabelled_features.csv")

## Original cell 21 (zero-based index)

In [ ]:
def check_data_health(df, name="Dataset"):
    print(f"--- Health Check: {name} ---")
    print(f"Shape: {df.shape}")

    # Check Missing Values
    na_count = df.isna().sum().sum()
    print(f"Total NaNs: {na_count}")
    if na_count > 0:
        print(df.isna().sum()[df.isna().sum() > 0])

    # Check High Values
    num_cols = df.select_dtypes(include=[np.number]).columns
    inf_count = np.isinf(df[num_cols]).sum().sum()
    print(f"Total Infs: {inf_count}")

    # Check Duplicates
    dupes = df.duplicated(subset=['txid']).sum()
    print(f"Duplicate TXIDs: {dupes}")
    print("")

check_data_health(final_train, "Train")
check_data_health(final_test, "Test")
check_data_health(final_valid, "Validation")   

## Original cell 25 (zero-based index)

In [ ]:
# We first ensure to use a clean dataset
df_eda = final_train.copy()

# Label Distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
label_counts = df_eda['label'].value_counts()
sns.barplot(x=label_counts.index, y=label_counts.values, ax=axes[0], palette='viridis', hue=label_counts.index, legend=False)
axes[0].set_title('Transaction Counts by Service', fontsize=14, fontweight='bold')
axes[0].set_yscale('log') # to see the lower classes
axes[0].set_ylabel('Count (Log Scale)')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v, f"{v:,}", ha='center', va='bottom', fontweight='bold')
label_pct = (label_counts / label_counts.sum() * 100)
explode = [0.1 if x < 5 else 0 for x in label_pct] 
axes[1].pie(label_pct.values, labels=label_pct.index, autopct='%1.1f%%', 
            colors=sns.color_palette("viridis", len(label_pct)), 
            explode=explode, startangle=140)
axes[1].set_title('Market Share (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()


## Original cell 28 (zero-based index)

In [ ]:
# Key Feature Histograms
features_to_plot = [
    'n_inputs', 'n_outputs', 'unique_output_ratio',
    'entropy', 'p2wpkh_ratio', 'fee_rate'
]

fig, axes = plt.subplots(2, 3, figsize=(18,10))
axes = axes.ravel()
for idx, feature in enumerate(features_to_plot):
    if feature in df_eda.columns:
        for label in df_eda['label'].unique():
            subset = df_eda[df_eda['label'] == label]
            axes[idx].hist(subset[feature], alpha=0.6, label=label, bins=30, density=True)
        axes[idx].set_title(f'{feature} Distribution', fontsize=12, fontweight='bold')
        axes[idx].set_yscale('log') # To see better
        axes[idx].legend(fontsize=8)
    else:
        axes[idx].set_visible(False)
plt.tight_layout()
plt.show()

## Original cell 31 (zero-based index)

In [ ]:
# Cluster analysis
plt.figure(figsize=(14, 8))
sns.scatterplot(
    data=final_train, 
    x="n_outputs", 
    y="entropy", 
    hue="label", 
    alpha=0.6,
    palette="bright"
)
plt.title("Behavioral Clustering : Entropy vs Outputs", fontsize=15, fontweight='bold')
plt.xscale('log')
plt.ylabel("Entropy (Randomness)")
plt.xlabel("Number of Outputs (Log Scale)")
plt.legend(bbox_to_anchor=(1.05, 1), loc=2)
plt.show()

## Original cell 34 (zero-based index)

In [ ]:
# Correlation Matrix
feature_cols = df_eda.select_dtypes(include=[np.number]).columns.tolist()
cols_to_exclude = ['txid', 'label', 'timestamps']
feature_cols = [c for c in feature_cols if c not in cols_to_exclude]

corr_matrix = df_eda[feature_cols].corr()

plt.figure(figsize=(14,10))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=False, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.show()

## Original cell 39 (zero-based index)

In [ ]:
# Drop columns that are in a wrong format or highly correlated
cols_to_drop = ['txid', 'label', 'timestamps', 'total_output_btc','mean_output_vals', 'fee_per_input','complexity_score', 'log_volume']

X = final_train.drop(columns=cols_to_drop, errors='ignore')
y = final_train['label']

#Label Encoding
le = LabelEncoder()
y_encoded = le.fit_transform(y)
label_mapping = dict(zip(le.transform(le.classes_), le.classes_))
print(f"Target Mapping : {label_mapping}")

# Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []
fold = 1
for train_idx, val_idx in skf.split(X, y_encoded):
    X_fold_train, X_fold_val = X.iloc[train_idx], X.iloc[val_idx]
    y_fold_train, y_fold_val = y_encoded[train_idx], y_encoded[val_idx]

    weights_fold = compute_sample_weight(class_weight='balanced', y=y_fold_train)

    # Temporary model
    model_cv = XGBClassifier(
        n_estimators=100, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        objective='multi:softprob', n_jobs=-1, random_state=42, verbosity=0
    )

    model_cv.fit(X_fold_train, y_fold_train, sample_weight=weights_fold, verbose=False)

    preds = model_cv.predict(X_fold_val)
    score = f1_score(y_fold_val, preds, average='macro')
    cv_scores.append(score)

    print(f" > Fold {fold}/5 : F1-Score = {score:.4f}")
    fold+=1

print(f"Average stability: {np.mean(cv_scores):.4f} (± {np.std(cv_scores):.4f})")


## Original cell 41 (zero-based index)

In [ ]:
# First split data
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

# Apply weights to take minority classes into account
sample_weights = compute_sample_weight(
    class_weight='balanced',
    y=y_train
)


## Original cell 43 (zero-based index)

In [ ]:
# Search hyperparameters
param_dist = {      
    'max_depth': [6,7],               
    'learning_rate': [0.005, 0.01],   
    'subsample': [0.8, 0.9],              
    'colsample_bytree': [ 0.8,0.9]        
}

search = RandomizedSearchCV(
    estimator=XGBClassifier(n_estimators=800, objective='multi:softprob', random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=10,
    scoring='f1_macro', 
    cv=3,             
    verbose=1,
    random_state=42
)

print(" Searching for optimal hyperparameters...")
search.fit(X_train, y_train, sample_weight=sample_weights)

best_params = search.best_params_
best_model = search.best_estimator_

print(f"\n Best parameters found: {best_params}")
print(f" Associated validation score: {search.best_score_:.4f}")

## Original cell 44 (zero-based index)

In [ ]:
# Model Configuration and Training 
model = XGBClassifier(
    **best_params,
    n_estimators=10000, # Numbers of trees
    objective='multi:softprob', # More than 2 clusters
    eval_metric='mlogloss', # To punish mistakes
    early_stopping_rounds=20, # To stop before overfitting
    random_state=42, # To keep result
    n_jobs=-1 # Use all CPU cores
)

print("\ Training model...")
model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=200)
print("\ Training Complete.")


## Original cell 46 (zero-based index)

In [ ]:
# Prediction
y_pred = model.predict(X_val)
y_pred_proba = model.predict_proba(X_val)
print(classification_report(y_val, y_pred, target_names=le.classes_))

## Original cell 47 (zero-based index)

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_val, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
plt.figure(figsize=(7,5))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('Normalized Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right') 
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Original cell 48 (zero-based index)

In [ ]:
# ROC Curve
y_val_binarized = label_binarize(y_val, classes=range(len(le.classes_)))
fpr = {}
tpr = {}
roc_auc = {}

plt.figure(figsize=(10,8))
colors = plt.cm.Set2(range(len(le.classes_)))

for i, color in zip(range(len(le.classes_)), colors):
    fpr[i], tpr[i], _ = roc_curve(y_val_binarized[:, i], y_pred_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    plt.plot(fpr[i], tpr[i], color=color, lw=2, label=f'{le.classes_[i]} (AUC = {roc_auc[i]:.4f})')

plt.plot([0, 1], [0, 1], 'k', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves ', fontsize=14, fontweight='bold')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

print(f"Global Macro-Average AUC: {np.mean(list(roc_auc.values())):.4f}")

## Original cell 52 (zero-based index)

In [ ]:
X_shap_sample = X_val.sample(1000, random_state=42)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_shap_sample)

plt.figure(figsize=(10,6))
shap.summary_plot(shap_values, X_shap_sample, plot_type="bar", class_names=le.classes_)
plt.show()

## Original cell 54 (zero-based index)

In [ ]:
# Shap Per Class
for i, class_name in enumerate(le.classes_):
    print(f" Shap Analysis for : {class_name}")
    vals = shap_values[:, :, i]
    plt.figure(figsize=(8,5))
    shap.summary_plot(vals, X_shap_sample, show=False, max_display=10, plot_type="dot")
    plt.title(f'Top Features for {class_name}', fontsize=12, fontweight='bold')
    plt.show()

## Original cell 57 (zero-based index)

In [ ]:
# Set validation data
X_submission = final_valid.drop(columns=cols_to_drop, errors='ignore')
X_submission = X_submission[X_train.columns] #Validation set predictions

# Batch
probs = model.predict_proba(X_submission)
preds_idx = model.predict(X_submission)
preds_labels = le.inverse_transform(preds_idx)

# DataFrame of submission
submission = pd.DataFrame()
submission['txid'] = final_valid['txid']
probs_df = pd.DataFrame(probs, columns=le.classes_)
rename_map = {
    'WasabiWallet.io': 'pred_wasabi',
    'Bitcoin Fog': 'pred_bitcoin_fog',
    'SamouraiWallet.com': 'pred_samourai',
    'JoinMarket': 'pred_join_market',
    'other': 'pred_other'
}
probs_df = probs_df.rename(columns=rename_map)
submission = pd.concat([submission, probs_df], axis=1)
submission['label_predicted'] = preds_labels
filename = "validation_pred.csv"
submission.to_csv(filename, index=False)
print(f" File Saved successfully: {filename}")